# Fieldwise crop-yield model

This notebook trains the XGBoost yield model from `data/master_dataset.csv` and can expose it to the local Next.js frontend through a temporary ngrok URL. The same trainer and FastAPI service run locally without Colab.

**Colab:** Run the setup cell, then the API cell. In Colab, add an `NGROK_AUTHTOKEN` secret (key icon in the left sidebar) before starting the API. Then set the printed `/predict-yield` URL in `frontend/.env.local` and restart Next.js.

**Local is usually simpler:** follow the “Run the model locally” section in `frontend/README.md`; it does not need ngrok or Colab.

The endpoint predicts yield in bushels per acre. Its Low/Medium/High risk label compares the estimate with the model training set's crop-specific 25th and 75th yield percentiles.


In [ ]:
# Clone the project into this Colab runtime and install the model dependencies.
from pathlib import Path
import os, subprocess, sys

repo = Path('/content/DSU-Hackathon-2026-Team-RootCause')
if not repo.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/ms-cc-org/DSU-Hackathon-2026-Team-RootCause.git',
        str(repo),
    ], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'model_api/requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'model_api.train_model'], check=True)


## Start the temporary Colab API

Keep the API cell running while using the frontend. The ngrok URL is temporary and changes when the Colab runtime restarts. Do not paste an ngrok token into the notebook.


In [ ]:
# Start FastAPI on port 8000 and expose it to the browser via your Colab secret.
from google.colab import userdata
from pyngrok import ngrok
import subprocess, sys, time, urllib.request

ngrok_token = userdata.get('NGROK_AUTHTOKEN')
if not ngrok_token:
    raise RuntimeError('Add NGROK_AUTHTOKEN under Colab Secrets, grant this notebook access, and rerun this cell.')
ngrok.set_auth_token(ngrok_token)
api_process = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'model_api.api:app',
    '--host', '0.0.0.0', '--port', '8000',
])
time.sleep(3)
tunnel = ngrok.connect(8000, 'http')
print('Set NEXT_PUBLIC_PREDICTION_API_URL to:', tunnel.public_url + '/predict-yield')
print('Health check:', urllib.request.urlopen(tunnel.public_url + '/').read().decode())


## Stop the temporary API

Run this cell when finished.


In [ ]:
ngrok.disconnect(tunnel.public_url)
api_process.terminate()
